# Bronze enrichment audit
Read-only checks against the existing local PostgreSQL lake. Run from the installed project environment. See ENRICHMENT_AUDIT.md for interpretation.

In [1]:
import pandas as pd
import psycopg
from dengue_st_diagnostics.research_panel import fetch_cells, extract, build_features

with psycopg.connect(dbname="indonesia_vector_lake") as connection:
    source_cells = pd.DataFrame(connection.execute("""
        select payload->>'source', count(*), sum((payload->>'nonempty_cells')::bigint)
        from bronze.record b join metadata.dataset d using(dataset_id)
        where d.dataset_name='public_tabular_tables'
        group by 1 order by 3 desc
    """).fetchall(), columns=["source", "tables", "cells"])
    observations, issues = extract(fetch_cells(connection))
features = build_features(observations)
display(source_cells)
display(observations.groupby("quality_status").size())
display(issues.groupby("reason").size())
display(features.groupby("target_year").size())
assert features.predictor_year.eq(features.target_year - 1).all()
assert not observations.observation_key.duplicated().any()


                    source  tables    cells
0                   zenodo     422  1391144
1               data.go.id    1848   440919
2  researchdata.jcu.edu.au       7   129941

quality_status
review_required    2720
valid              2992
dtype: int64

reason
fatality_rate_disagrees_with_cases_deaths     16
incidence_disagrees_with_cases_population    122
missing_or_out_of_range                       70
dtype: int64

target_year
2018    14
2019    19
2020    15
2021    11
2022     9
2023     8
2024    10
dtype: int64